# Stable-Baselines3 Atari Mini Project — Report

**Course:** Graduate AI — Fresno State, Spring 2026  
**Submission:** April 26, 2026  
**Environments:** Pong (`ALE/Pong-v5`), Space Invaders (`ALE/SpaceInvaders-v5`)  
**Algorithms:** DQN (required baseline), A2C, PPO  
**Training budget:** 10M timesteps per algorithm per environment  
**Hardware:** NVIDIA RTX 3060 Ti, Windows 11, PyTorch 2.6 (CUDA 12.4)

## 1. Overview

This project trains and compares three reinforcement learning algorithms — **DQN**, **A2C**, and **PPO** — on two Atari environments using [Stable-Baselines3](https://stable-baselines3.readthedocs.io/). Each agent is trained for 10 million timesteps with model checkpoints saved periodically. Agents are evaluated at key milestones using 20 held-out deterministic episodes, measuring mean and standard deviation of episode reward.

## 2. Algorithms

### DQN (Deep Q-Network) — Off-policy, value-based
DQN learns a Q-function mapping (state, action) → expected return, using a **replay buffer** to decorrelate transitions and a **target network** for stable updates. Because it stores and reuses past transitions, it is sample-efficient but requires more memory and a warm-up period before learning starts.

**Space Invaders config (`dqn_si_default`):** lr=1e-4, batch=64, buffer=500k, exploration=20%, target_update=1000, γ=0.99

### A2C (Advantage Actor-Critic) — On-policy, actor-critic
A2C maintains two networks: an **actor** (policy) and a **critic** (value function). It updates using the advantage A(s,a) = r + γV(s') − V(s), which reduces variance compared to pure policy gradients. Updates are synchronous across 8 parallel environments and begin immediately — no replay buffer warm-up.

**Space Invaders config (`a2c_si_default`):** lr=7e-4, 8 envs, n_steps=10, ent_coef=0.02, γ=0.99

### PPO (Proximal Policy Optimization) — On-policy, actor-critic
PPO improves on A2C by **clipping the policy update ratio** (clip_range=0.1), preventing destructively large gradient steps. It collects longer rollouts (n_steps=256) across 8 parallel environments before each update, making it more stable than A2C at the cost of slightly slower initial learning.

**Space Invaders config (`ppo_si_default`):** lr=2.5e-4, 8 envs, n_steps=256, batch=256, clip=0.1, ent_coef=0.01, GAE λ=0.95, γ=0.99

## 3. Evaluation Setup

Each checkpoint is evaluated on **20 held-out episodes** with a deterministic policy (no exploration noise). The environment uses standard SB3 Atari preprocessing:
- Grayscale + resize to 84×84 pixels
- Frame stacking: 4 consecutive frames → (4, 84, 84) observation
- Episode termination on life loss during training; full-game evaluation at test time

**Metrics:** mean episode reward and standard deviation across 20 episodes.

In [ ]:
import json, os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

EVAL_FILES = {
    "Space Invaders": {
        "DQN": "../results/dqn_si_eval.json",
        "A2C": "../results/a2c_si_eval.json",
        "PPO": "../results/ppo_si_eval.json",
    },
    "Pong": {
        "DQN": "../results/dqn_pong_eval.json",
        "A2C": "../results/a2c_pong_eval.json",
        "PPO": "../results/ppo_pong_eval.json",
    },
}
COLORS = {"DQN": "#1f77b4", "A2C": "#ff7f0e", "PPO": "#2ca02c"}
FINAL_TS = 10_000_000

def parse_timestep(path):
    name = os.path.basename(path)
    if "final" in name:
        return FINAL_TS
    m = re.search(r"_(\d+)_steps", name)
    return int(m.group(1)) if m else 0

def load(path):
    with open(path) as f:
        raw = json.load(f)
    seen = {}
    for e in raw:
        ts = parse_timestep(e["checkpoint"])
        seen[ts] = {"timestep": ts, "mean": e["mean_reward"], "std": e["std_reward"]}
    return sorted(seen.values(), key=lambda r: r["timestep"])

all_data = {}
for env, files in EVAL_FILES.items():
    all_data[env] = {}
    for algo, path in files.items():
        if os.path.exists(path):
            all_data[env][algo] = load(path)
        else:
            print(f"[missing] {path}")

print("Loaded:", {env: list(algos.keys()) for env, algos in all_data.items()})

## 4. Results Tables

In [ ]:
def fmt_ts(x):
    return f"{x//1_000_000}M" if x >= 1_000_000 else f"{x//1_000}k"

for env, algos in all_data.items():
    records = []
    for algo, rows in algos.items():
        for r in rows:
            records.append({"Algorithm": algo, "Timesteps": fmt_ts(r["timestep"]),
                            "Mean Reward": round(r["mean"], 1), "Std": round(r["std"], 1)})
    df = pd.DataFrame(records)
    print(f"\n=== {env} ===")
    display(df)

## 5. Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (env_name, algos) in zip(axes, all_data.items()):
    for algo, rows in algos.items():
        xs = [r["timestep"] for r in rows]
        ys = [r["mean"] for r in rows]
        errs = [r["std"] for r in rows]
        ax.plot(xs, ys, marker="o", label=algo, color=COLORS[algo])
        ax.fill_between(xs,
                        [y - e for y, e in zip(ys, errs)],
                        [y + e for y, e in zip(ys, errs)],
                        alpha=0.15, color=COLORS[algo])
    if env_name == "Pong":
        ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    ax.set_xlabel("Training Timesteps")
    ax.set_ylabel("Mean Episode Reward (20 episodes)")
    ax.set_title(f"DQN vs A2C vs PPO — {env_name} (10M steps)")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f"{int(x/1e6)}M" if x >= 1e6 else f"{int(x/1e3)}k"))
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.tight_layout()
os.makedirs("../report", exist_ok=True)
fig.savefig("../report/learning_curves.png", dpi=150)
plt.show()
print("Saved → report/learning_curves.png")

## 6. Discussion

### 6.1 Pong — DQN dominates

Pong rewards range from −21 (losing every point) to +21 (winning every point). At 100k steps all three algorithms scored −21, showing no learning yet. **DQN** was the clear winner: it crossed the zero break-even line at ~5M steps (+5.3) and reached **+9.2 at 10M** — consistently winning more games than it loses. **PPO** improved steadily but slowly, finishing at −3.1 at 10M. **A2C** struggled throughout the run, ending at −8.4, still heavily losing.

DQN's dominance on Pong makes sense: Pong has a simple, deterministic structure where the replay buffer excels. Storing and replaying high-quality transitions (rallies where the agent scored) is very efficient for a game with only two meaningful actions (up/down).

### 6.2 Space Invaders — All three learned, DQN peaks highest

Space Invaders is significantly harder than Pong: 6 meaningful actions, sparse rewards (points only on alien hits), and a complex threat model (shield usage, alien movement patterns). All three algorithms showed meaningful learning:

- **A2C** started fastest — 363 at 100k, the best early score — because on-policy methods update immediately without a warm-up period. However it showed high variance and a dip at 1M before recovering.
- **PPO** was the most stable early on (±40 std at 100k vs ±213 for A2C), reflecting the clipped update objective preventing erratic policy changes.
- **DQN** started slowest (217 at 100k — still in exploration phase) but grew steadily and achieved the highest final reward: **1064 ± 485 at 10M**. A2C reached 914 and PPO 861.

### 6.3 Off-policy vs On-policy tradeoffs

| Property | DQN (off-policy) | A2C / PPO (on-policy) |
|---|---|---|
| Updates from | Replay buffer (past data) | Current rollouts only |
| Warm-up needed | Yes (50k steps before learning) | No — updates immediately |
| Sample efficiency | Higher (reuses data) | Lower (discards after update) |
| Stability | High (target network) | A2C: medium, PPO: high |
| Parallelism | 1 env | 8 envs |

DQN's replay buffer becomes a decisive advantage at long time horizons (10M steps) because it can revisit rare but informative experiences (e.g., encounters with the bonus UFO in Space Invaders). On-policy methods must re-learn from scratch each rollout.

### 6.4 Variance and reliability

All algorithms show high standard deviation in Space Invaders (±300–500 at 10M), reflecting the stochastic nature of the environment. DQN's variance actually grows with performance — a sign that it has learned multiple strategies of varying quality rather than a single dominant one. PPO maintains tighter variance throughout, suggesting more consistent (if lower peak) behavior.

### 6.5 Checkpoint progression

Gameplay videos at each 100k checkpoint (saved in `videos/space_invaders/`) show clear visual progression:
- **Early (100k–500k):** Agent fires almost randomly, rarely dodges alien shots
- **Mid (1M–5M):** Agent begins targeting alien clusters and using shields for cover
- **Late (5M–10M):** Agent shows consistent left-right sweeping patterns and improved survival time

### 6.6 Conclusion

DQN, the required baseline, proved to be the strongest algorithm across both environments at 10M timesteps. Its experience replay and target network make it particularly well-suited to Atari's visual complexity at longer training runs. A2C showed faster initial learning on Space Invaders but did not sustain that advantage. PPO offered the best stability-performance tradeoff among the on-policy methods.

All three algorithms successfully learned non-trivial strategies from raw pixels — a striking result that highlights the power of deep reinforcement learning for vision-based sequential decision making.

## 7. Hyperparameter Summary

| Parameter | DQN | A2C | PPO |
|---|---|---|---|
| Learning rate | 1e-4 | 7e-4 | 2.5e-4 |
| Parallel envs | 1 | 8 | 8 |
| Gamma (γ) | 0.99 | 0.99 | 0.99 |
| Batch size | 64 | n/a | 256 |
| Replay buffer | 500k | n/a | n/a |
| n_steps per update | n/a | 10 | 256 |
| Entropy coefficient | n/a | 0.02 | 0.01 |
| Exploration | ε-greedy (20%) | entropy bonus | clipped ratio + entropy |